<a href="https://colab.research.google.com/github/4541kiet-divakar/Divakar4541/blob/main/Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===== CELL 1: create project files =====
import os, pathlib
os.makedirs('/content/grade-curving-A8', exist_ok=True)
P = pathlib.Path('/content/grade-curving-A8')

(P/'grade_curving.py').write_text(r'''"""Student Grade Curving System (A8)
Model: y = a*x + b per professor, unknowns (a, b, d), 3x3 system via Gauss elimination.
Assumptions: a>0, population std (ddof=0), default target = pooled mean/std.
Paste into ONE Colab cell and run."""

# ======================================================================
# IMPORTS
# ======================================================================
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid", context="notebook")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)

# ======================================================================
# INPUTS
# ======================================================================
# ---------- OPTION A: built-in benchmark dataset (realistic, reproducible) ----------
# Prof A = hard paper, Prof B = medium, Prof C = easy paper. 30 students each.
USE_BENCHMARK = True

# ---------- OPTION B: your own data (set USE_BENCHMARK = False) ----------
MY_SCORES = {
    "Prof A": [45, 52, 60, 38, 70, 55],
    "Prof B": [60, 72, 65, 80, 58, 69],
    "Prof C": [75, 88, 82, 91, 70, 85],
}

# ---------- Targets ----------
TARGET_MEAN = None   # None -> pooled mean of all students
TARGET_STD  = None   # None -> pooled std-dev of all students
CAP = 100.0          # maximum allowed score (boundary condition)

if USE_BENCHMARK:
    rng = np.random.default_rng(42)                      # fixed seed -> same data every run
    params = {"Prof A": (58, 12), "Prof B": (68, 9), "Prof C": (78, 7)}   # (mean, sd) of raw marks
    SCORES = {k: np.clip(np.round(rng.normal(mu, sd, 30)), 0, 100) for k, (mu, sd) in params.items()}
else:
    SCORES = {k: np.array(v, dtype=float) for k, v in MY_SCORES.items()}

assert len(SCORES) == 3, "This project expects exactly 3 professors."
pooled = np.concatenate(list(SCORES.values()))
mu_T  = TARGET_MEAN if TARGET_MEAN is not None else pooled.mean()
sig_T = TARGET_STD  if TARGET_STD  is not None else pooled.std()   # ddof=0
print(f"Target mean μ_T = {mu_T:.3f}   Target std σ_T = {sig_T:.3f}   Cap = {CAP}")

# ======================================================================
# MATH ENGINE
# ======================================================================
def section_stats(x):
    """Raw mean, population std-dev and top score of one professor's section."""
    return x.mean(), x.std(), x.max()

def build_system(m, s, x_max, mu_T, sig_T, cap):
    """
    Unknown vector u = [a, b, d]
      Eq1 (mean)    : m*a    + 1*b + 0*d = mu_T
      Eq2 (variance): s*a    + 0*b + 0*d = sig_T      (uses a>0 so sqrt(Var) is linear)
      Eq3 (boundary): x_max*a + 1*b + 1*d = cap
    """
    A = np.array([[m,     1.0, 0.0],
                  [s,     0.0, 0.0],
                  [x_max, 1.0, 1.0]])
    rhs = np.array([mu_T, sig_T, cap])
    return A, rhs

def gauss_eliminate(A, rhs, verbose=False):
    """
    Gauss elimination with partial pivoting, written out step by step.
    Forward elimination -> upper-triangular matrix; then back-substitution.
    """
    M = np.hstack([A.astype(float), rhs.reshape(-1, 1).astype(float)])   # augmented [A | b]
    n = len(rhs)
    if verbose: print("Augmented matrix [A | b]:\n", M, "\n")
    for k in range(n - 1):
        # Partial pivoting: swap in the row with the largest |entry| in column k
        p = k + np.argmax(np.abs(M[k:, k]))
        if abs(M[p, k]) < 1e-12:
            raise ValueError("Singular system (zero pivot): e.g. all scores identical -> std = 0.")
        if p != k:
            M[[k, p]] = M[[p, k]]
            if verbose: print(f"Swap R{k+1} <-> R{p+1}")
        # Eliminate entries below the pivot:  R_i <- R_i - (M[i,k]/M[k,k]) * R_k
        for i in range(k + 1, n):
            f = M[i, k] / M[k, k]
            M[i, k:] -= f * M[k, k:]
            if verbose: print(f"R{i+1} <- R{i+1} - ({f:.4f})*R{k+1}")
        if verbose: print(M, "\n")
    if abs(M[n-1, n-1]) < 1e-12:
        raise ValueError("Singular system (zero pivot).")
    # Back-substitution (solve from the last row upward)
    u = np.zeros(n)
    for i in range(n - 1, -1, -1):
        u[i] = (M[i, n] - M[i, i+1:n] @ u[i+1:]) / M[i, i]
    return u

def solve_with_sympy(A, rhs):
    """Independent exact check using SymPy."""
    return np.array(sp.Matrix(A.tolist()).LUsolve(sp.Matrix(rhs.tolist())), dtype=float).flatten()

def curve_all(scores, mu_T, sig_T, cap, verbose_first=True):
    results, curved = {}, {}
    for idx, (name, x) in enumerate(scores.items()):
        m, s, xmax = section_stats(x)
        A, rhs = build_system(m, s, xmax, mu_T, sig_T, cap)
        if verbose_first and idx == 0:
            print(f"=== Gauss elimination for {name} ===")
        a, b, d = gauss_eliminate(A, rhs, verbose=(verbose_first and idx == 0))
        results[name] = dict(m=m, s=s, xmax=xmax, a=a, b=b, d=d, A=A, rhs=rhs)
        curved[name] = a * x + b
    return results, curved

# ======================================================================
# RUN & OUTPUT DISPLAY
# ======================================================================
results, curved = curve_all(SCORES, mu_T, sig_T, CAP)

# --- The 3x3 system for each professor, written as equations ---
print("\n=== Systems built ===")
for name, r in results.items():
    print(f"{name}:  {r['m']:.3f}a + b = {mu_T:.3f} | {r['s']:.3f}a = {sig_T:.3f} | {r['xmax']:.0f}a + b + d = {CAP:.0f}")

# --- Solutions table ---
sol = pd.DataFrame({n: {"a (scale)": r["a"], "b (shift)": r["b"], "d (headroom)": r["d"]}
                    for n, r in results.items()}).T
print("\n=== Solution of each 3x3 system ===")
display(sol)

# --- Before / after statistics ---
rows = []
for n in SCORES:
    rows.append({"Professor": n,
                 "Raw mean": SCORES[n].mean(), "Raw std": SCORES[n].std(),
                 "Curved mean": curved[n].mean(), "Curved std": curved[n].std(),
                 "Raw max": SCORES[n].max(), "Curved max": curved[n].max(),
                 "Cap OK?": "✅" if results[n]["d"] >= -1e-9 else "⚠️ exceeds cap"})
summary = pd.DataFrame(rows).set_index("Professor")
print("\n=== Before vs After ===")
display(summary)

if any(r["d"] < 0 for r in results.values()):
    print("⚠️ d < 0 for some section: top scorer exceeds the cap. Lower the targets or clip (clipping breaks exact matching).")

# ======================================================================
# CHARTS
# ======================================================================
colors = {"Prof A": "#e76f51", "Prof B": "#2a9d8f", "Prof C": "#457b9d"}
long = lambda data, label: pd.DataFrame([(n, v, label) for n, arr in data.items() for v in arr],
                                        columns=["Professor", "Score", "Stage"])
df = pd.concat([long(SCORES, "Before (raw)"), long(curved, "After (curved)")])

fig, ax = plt.subplots(2, 2, figsize=(14, 10))

# 1) Distributions before vs after (boxplot + points)
sns.boxplot(data=df, x="Professor", y="Score", hue="Stage", palette=["#bbbbbb", "#2a9d8f"], ax=ax[0, 0])
ax[0, 0].axhline(mu_T, color="k", ls="--", lw=1, label="Target mean")
ax[0, 0].set_title("Score spread: before vs after curving"); ax[0, 0].legend(fontsize=8)

# 2) Density curves: raw sections differ, curved sections overlap
for n in SCORES:
    sns.kdeplot(SCORES[n], ax=ax[0, 1], color=colors[n], ls="--", lw=1.5, label=f"{n} raw")
    sns.kdeplot(curved[n], ax=ax[0, 1], color=colors[n], lw=2.5, label=f"{n} curved")
ax[0, 1].set_title("Density: curved distributions align"); ax[0, 1].legend(fontsize=8); ax[0, 1].set_xlabel("Score")

# 3) The fitted lines y = a x + b
xs = np.linspace(0, 100, 100)
for n, r in results.items():
    ax[1, 0].plot(xs, r["a"] * xs + r["b"], color=colors[n], lw=2.5, label=f"{n}: y={r['a']:.2f}x{r['b']:+.1f}")
ax[1, 0].plot(xs, xs, "k:", label="No curve (y=x)")
ax[1, 0].axhline(CAP, color="red", ls="--", lw=1, label="Cap")
ax[1, 0].set_title("Curving functions (Gauss-elimination solution)")
ax[1, 0].set_xlabel("Raw score"); ax[1, 0].set_ylabel("Curved score"); ax[1, 0].legend(fontsize=8)

# 4) Mean & std before/after (should collapse to the target)
stats = pd.DataFrame({
    "Professor": list(SCORES) * 2,
    "Stage": ["Raw"] * 3 + ["Curved"] * 3,
    "Mean": [SCORES[n].mean() for n in SCORES] + [curved[n].mean() for n in SCORES],
    "Std":  [SCORES[n].std()  for n in SCORES] + [curved[n].std()  for n in SCORES]})
w = 0.35; x = np.arange(3)
for j, (stage, c) in enumerate([("Raw", "#bbbbbb"), ("Curved", "#2a9d8f")]):
    sub = stats[stats.Stage == stage]
    ax[1, 1].bar(x + (j - .5) * w, sub.Mean, w, color=c, label=f"{stage} mean")
    ax[1, 1].errorbar(x + (j - .5) * w, sub.Mean, yerr=sub.Std, fmt="none", ecolor="k", capsize=4)
ax[1, 1].axhline(mu_T, color="k", ls="--", lw=1)
ax[1, 1].set_xticks(x); ax[1, 1].set_xticklabels(list(SCORES))
ax[1, 1].set_title("Mean ± std per professor (error bars = std)"); ax[1, 1].legend(fontsize=8)

plt.tight_layout(); plt.show()

# ======================================================================
# BENCHMARK & VERIFICATION TESTS
# ======================================================================
print("Running verification tests...\n")

# Test 1: Gauss elimination == NumPy solver == SymPy exact solver (all 3 professors)
for n, r in results.items():
    u_g = np.array([r["a"], r["b"], r["d"]])
    u_np = np.linalg.solve(r["A"], r["rhs"])
    u_sp = solve_with_sympy(r["A"], r["rhs"])
    assert np.allclose(u_g, u_np) and np.allclose(u_g, u_sp), f"Solver mismatch for {n}"
print("✅ Test 1: Gauss == numpy.linalg.solve == SymPy")

# Test 2: curved mean and std equal the targets for every professor
for n in SCORES:
    assert abs(curved[n].mean() - mu_T) < 1e-9 and abs(curved[n].std() - sig_T) < 1e-9
print("✅ Test 2: all 3 curved sections have identical mean and std")

# Test 3: ranking preserved (a > 0)
for n in SCORES:
    assert results[n]["a"] > 0 and np.array_equal(np.argsort(SCORES[n], kind="stable"),
                                                  np.argsort(curved[n], kind="stable"))
print("✅ Test 3: student ranking within each section is preserved")

# Test 4: hand-checkable case. Scores [0,100]: m=50, s=50. Target (50,50), cap 100 -> a=1, b=0, d=0
x = np.array([0., 100.])
A, rhs = build_system(*section_stats(x), 50, 50, 100)
assert np.allclose(gauss_eliminate(A, rhs), [1, 0, 0])
print("✅ Test 4: hand-checked case gives a=1, b=0, d=0")

# Test 5: singular case is caught (all identical scores -> std 0)
try:
    A, rhs = build_system(*section_stats(np.array([60., 60., 60.])), 70, 10, 100)
    gauss_eliminate(A, rhs); print("❌ should have failed")
except ValueError as e:
    print("✅ Test 5: degenerate data detected ->", e)

print("\nAll tests passed 🎉")

''')

(P/'index.html').write_text(r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Student Grade Curving System (A8)</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
:root{--bg:#f6f8fb;--card:#fff;--tx:#1d2733;--mut:#6b7785;--bd:#e1e6ed;--ac:#2a9d8f}
@media(prefers-color-scheme:dark){:root{--bg:#10151b;--card:#1a222b;--tx:#e8edf2;--mut:#93a0ae;--bd:#2c3744}}
*{box-sizing:border-box}
body{margin:0;font-family:system-ui,Segoe UI,Roboto,sans-serif;background:var(--bg);color:var(--tx);line-height:1.5}
header{padding:28px 20px;background:linear-gradient(135deg,#2a9d8f,#457b9d);color:#fff}
header h1{margin:0;font-size:1.6rem} header p{margin:6px 0 0;opacity:.9}
main{max-width:1100px;margin:auto;padding:16px}
.card{background:var(--card);border:1px solid var(--bd);border-radius:14px;padding:16px;margin-bottom:16px}
h2{margin:0 0 10px;font-size:1.1rem}
.grid3{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:12px}
.grid2{display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:16px}
label{font-size:.85rem;color:var(--mut);display:block;margin-bottom:4px}
textarea,input{width:100%;padding:8px;border:1px solid var(--bd);border-radius:8px;background:var(--bg);color:var(--tx);font:inherit}
textarea{min-height:90px}
.row{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-top:12px}
button{padding:10px 18px;border:0;border-radius:8px;background:var(--ac);color:#fff;font:inherit;font-weight:600;cursor:pointer;margin:12px 8px 0 0}
button.alt{background:transparent;color:var(--ac);border:1px solid var(--ac)}
table{width:100%;border-collapse:collapse;font-size:.9rem}
th,td{padding:6px 8px;border-bottom:1px solid var(--bd);text-align:right;white-space:nowrap}
th:first-child,td:first-child{text-align:left}
.scroll{overflow-x:auto}
pre{background:var(--bg);padding:10px;border-radius:8px;overflow-x:auto;font-size:.8rem}
.err{color:#d62828;font-weight:600}.ok{color:#2a9d8f;font-weight:600}
small,.note{color:var(--mut);font-size:.82rem}
canvas{max-width:100%}
</style>
</head>
<body>
<header><h1>📐 Student Grade Curving System (A8)</h1>
<p>3×3 system · Gauss elimination · y = a·x + b per professor</p></header>
<main>

<div class="card">
<h2>Model &amp; assumptions</h2>
<div class="note">Unknowns <b>(a, b, d)</b> per professor. &nbsp;① Mean: <code>m·a + b = μ<sub>T</sub></code> &nbsp;② Variance: <code>s·a = σ<sub>T</sub></code> (uses a &gt; 0) &nbsp;③ Boundary: <code>x<sub>max</sub>·a + b + d = CAP</code>.<br>
Assumptions: linear curve, a &gt; 0, population std-dev, default target = pooled mean/std of all students.</div>
</div>

<div class="card">
<h2>Inputs</h2>
<div class="grid3">
<div><label>Prof A scores (comma / space separated)</label><textarea id="s0"></textarea></div>
<div><label>Prof B scores</label><textarea id="s1"></textarea></div>
<div><label>Prof C scores</label><textarea id="s2"></textarea></div>
</div>
<div class="row">
<div><label>Target mean (blank = pooled)</label><input id="tm" type="number" step="any"></div>
<div><label>Target std (blank = pooled)</label><input id="ts" type="number" step="any"></div>
<div><label>Cap (max score)</label><input id="cap" type="number" step="any" value="100"></div>
</div>
<button id="run">Curve grades</button>
<button id="bench" class="alt">Load benchmark data</button>
<div id="msg" class="err"></div>
</div>

<div id="out" hidden>
<div class="card"><h2>Targets &amp; systems</h2><div id="targets"></div><div id="eqs" class="scroll"></div></div>
<div class="card"><h2>Solution (a, b, d)</h2><div class="scroll" id="sol"></div></div>
<div class="card"><h2>Before vs after</h2><div class="scroll" id="cmp"></div></div>
<div class="card"><h2>Charts</h2>
<div class="grid2">
<div><canvas id="c1"></canvas></div><div><canvas id="c2"></canvas></div>
<div><canvas id="c3"></canvas></div><div><canvas id="c4"></canvas></div>
</div></div>
<div class="card"><h2>Gauss elimination steps</h2><div id="steps"></div></div>
<div class="card"><h2>Verification</h2><div id="ver"></div></div>
</div>
</main>

<script>
//MATH-START
const mean = x => x.reduce((a,b)=>a+b,0)/x.length;
const std  = x => { const m=mean(x); return Math.sqrt(mean(x.map(v=>(v-m)**2))); }; // population std

// Unknowns [a,b,d]; rows: mean, variance(a>0 -> linear), boundary
function buildSystem(m,s,xmax,muT,sigT,cap){
  return {A:[[m,1,0],[s,0,0],[xmax,1,1]], rhs:[muT,sigT,cap]};
}

// Gauss elimination with partial pivoting; returns solution + readable steps
function gauss(A,rhs){
  const n=rhs.length, steps=[];
  const M=A.map((r,i)=>[...r,rhs[i]]);
  const show=t=>steps.push(t+"\n"+M.map(r=>r.map(v=>v.toFixed(4).padStart(10)).join(" ")).join("\n"));
  show("Augmented matrix [A | b]");
  for(let k=0;k<n-1;k++){
    let p=k; for(let i=k+1;i<n;i++) if(Math.abs(M[i][k])>Math.abs(M[p][k])) p=i;
    if(Math.abs(M[p][k])<1e-12) throw new Error("Singular system (zero pivot): e.g. all scores identical → std = 0.");
    if(p!==k){[M[k],M[p]]=[M[p],M[k]]; show(`Swap R${k+1} ↔ R${p+1}`);}
    for(let i=k+1;i<n;i++){
      const f=M[i][k]/M[k][k];
      for(let j=k;j<=n;j++) M[i][j]-=f*M[k][j];
      show(`R${i+1} ← R${i+1} − (${f.toFixed(4)})·R${k+1}`);
    }
  }
  if(Math.abs(M[n-1][n-1])<1e-12) throw new Error("Singular system (zero pivot).");
  const u=new Array(n).fill(0);                       // back-substitution
  for(let i=n-1;i>=0;i--){
    let s=M[i][n]; for(let j=i+1;j<n;j++) s-=M[i][j]*u[j];
    u[i]=s/M[i][i];
  }
  return {u,steps};
}

// Independent check: Cramer's rule (determinants)
const det3=m=>m[0][0]*(m[1][1]*m[2][2]-m[1][2]*m[2][1])-m[0][1]*(m[1][0]*m[2][2]-m[1][2]*m[2][0])+m[0][2]*(m[1][0]*m[2][1]-m[1][1]*m[2][0]);
function cramer(A,rhs){
  const D=det3(A);
  return [0,1,2].map(c=>det3(A.map((r,i)=>r.map((v,j)=>j===c?rhs[i]:v)))/D);
}

function curveAll(data,muT,sigT,cap){
  return data.map(x=>{
    const m=mean(x),s=std(x),xmax=Math.max(...x);
    const {A,rhs}=buildSystem(m,s,xmax,muT,sigT,cap);
    const {u,steps}=gauss(A,rhs);
    const [a,b,d]=u;
    return {m,s,xmax,a,b,d,A,rhs,steps,curved:x.map(v=>a*v+b)};
  });
}
//MATH-END

const names=["Prof A","Prof B","Prof C"], cols=["#e76f51","#2a9d8f","#457b9d"];
const $=id=>document.getElementById(id); let charts=[];

// seeded RNG so the benchmark is reproducible
function rng(seed){return()=>{seed|=0;seed=seed+0x6D2B79F5|0;let t=Math.imul(seed^seed>>>15,1|seed);t=t+Math.imul(t^t>>>7,61|t)^t;return((t^t>>>14)>>>0)/4294967296;}}
function loadBench(){
  const r=rng(42), norm=()=>Math.sqrt(-2*Math.log(1-r()))*Math.cos(2*Math.PI*r());
  [[58,12],[68,9],[78,7]].forEach(([mu,sd],i)=>{
    const v=Array.from({length:30},()=>Math.min(100,Math.max(0,Math.round(mu+sd*norm()))));
    $("s"+i).value=v.join(", ");
  });
}
const parse=t=>t.split(/[\s,;]+/).filter(Boolean).map(Number);
const f=(v,k=3)=>Number(v).toFixed(k);

function table(head,rows){return `<table><tr>${head.map(h=>`<th>${h}</th>`).join("")}</tr>${rows.map(r=>`<tr>${r.map(c=>`<td>${c}</td>`).join("")}</tr>`).join("")}</table>`;}

function hist(x){const b=new Array(10).fill(0);x.forEach(v=>{b[Math.max(0,Math.min(9,Math.floor(v/10)))]++});return b;}

function run(){
  $("msg").textContent="";
  try{
    const data=[0,1,2].map(i=>parse($("s"+i).value));
    data.forEach((d,i)=>{ if(d.length<2||d.some(isNaN)) throw new Error(`${names[i]}: enter at least 2 valid numbers.`); });
    const pooled=data.flat(), cap=parseFloat($("cap").value)||100;
    const muT=$("tm").value!==""?parseFloat($("tm").value):mean(pooled);
    const sigT=$("ts").value!==""?parseFloat($("ts").value):std(pooled);
    const R=curveAll(data,muT,sigT,cap);

    $("targets").innerHTML=`Target mean μ<sub>T</sub> = <b>${f(muT)}</b> &nbsp; Target std σ<sub>T</sub> = <b>${f(sigT)}</b> &nbsp; Cap = <b>${cap}</b>`;
    $("eqs").innerHTML=table(["Professor","① mean","② variance","③ boundary"],
      R.map((r,i)=>[names[i],`${f(r.m)}a + b = ${f(muT)}`,`${f(r.s)}a = ${f(sigT)}`,`${r.xmax}a + b + d = ${cap}`]));
    $("sol").innerHTML=table(["Professor","a (scale)","b (shift)","d (headroom)","Cap OK?"],
      R.map((r,i)=>[names[i],f(r.a,4),f(r.b,4),f(r.d,4),r.d>=-1e-9?'<span class="ok">✅</span>':'<span class="err">⚠️ exceeds cap</span>']));
    $("cmp").innerHTML=table(["Professor","Raw mean","Raw std","Curved mean","Curved std","Raw max","Curved max"],
      R.map((r,i)=>[names[i],f(r.m),f(r.s),f(mean(r.curved)),f(std(r.curved)),r.xmax,f(Math.max(...r.curved))]));
    $("steps").innerHTML=R.map((r,i)=>`<details ${i===0?"open":""}><summary><b>${names[i]}</b></summary><pre>${r.steps.join("\n\n")}</pre></details>`).join("");

    // verification
    const checks=[];
    R.forEach((r,i)=>{const c=cramer(r.A,r.rhs);checks.push([`${names[i]}: Gauss = Cramer's rule`,[r.a,r.b,r.d].every((v,k)=>Math.abs(

_IncompleteInputError: incomplete input (3321699633.py, line 245)